# S1 w4096 relu max

condition=two_point_poles  width=expanded_4096  act=relu  pool=max  epochs=2000  seed=0


In [ ]:
CONDITION = "two_point_poles"
HD, ACT, POOL = "expanded_4096", "relu", "max"
EPOCHS, SEED = 2000, 0


In [ ]:
import os, sys, glob, shutil, time, json
import numpy as np
import torch
assert torch.cuda.is_available(), "No GPU - accelerator must be GPU T4 x2"
print("GPU:", torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
_ = (torch.randn(256, 256, device="cuda") @ torch.randn(256, 256, device="cuda")).sum().item()
print("CUDA OK")

INPUT = "/kaggle/input"
cands = glob.glob(os.path.join(INPUT, "**", "toy_datasets"), recursive=True)
if not cands:
    raise RuntimeError("toy_datasets not found; saw " + str([r for r,d,f in os.walk(INPUT)][:30]))
ROOT = os.path.dirname(cands[0])
WORK = "/kaggle/working/data"
os.makedirs(WORK, exist_ok=True)
for n in ["toy_datasets", "ground_truth"]:
    if not os.path.exists(os.path.join(WORK, n)):
        shutil.copytree(os.path.join(ROOT, n), os.path.join(WORK, n))
hits = glob.glob(os.path.join(INPUT, "**", "experiment_architecture_combined.py"), recursive=True)
shutil.copy(hits[0], "/kaggle/working/experiment_architecture_combined.py")
DATA = os.path.join(WORK, "toy_datasets", CONDITION)
GT = os.path.join(WORK, "ground_truth", CONDITION)
OUT = "/kaggle/working/run"
print("data:", DATA, "| samples:", len(glob.glob(DATA + "/*.npy")))


In [ ]:
# Run in-process so the weight-init seed can be overridden without editing the
# training script (its torch.manual_seed(0) is hardcoded at line 578).
sys.path.insert(0, "/kaggle/working")
import experiment_architecture_combined as X

_orig_seed = torch.manual_seed
torch.manual_seed = lambda _ignored: _orig_seed(SEED)
print("weight-init seed overridden to", SEED)

sys.argv = ["run", "--traindata-path", DATA, "--testdata-path", DATA,
            "--outdir", OUT, "--device", "cuda", "--epochs", str(EPOCHS),
            "--batch-size", "2", "--mode", "full_grid",
            "--hidden-dims-subset", HD, "--activations-subset", ACT,
            "--poolings-subset", POOL]
t0 = time.time()
X.main()
torch.manual_seed = _orig_seed
print("elapsed hours:", round((time.time() - t0) / 3600, 3))


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

run_dir = glob.glob(os.path.join(OUT, "hd=*"))[0]
hist = np.genfromtxt(os.path.join(run_dir, "history.csv"), delimiter=",",
                     names=True, dtype=None, encoding=None)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for i, k in enumerate(["total", "chamfer", "pinns"]):
    ax[i].plot(hist[k]); ax[i].set_yscale("log")
    ax[i].set_title(k); ax[i].set_xlabel("epoch"); ax[i].grid(alpha=.3)
plt.suptitle(os.path.basename(run_dir) + "  seed=" + str(SEED))
plt.tight_layout(); plt.savefig("/kaggle/working/loss_curves.png", dpi=110)
plt.show()

n = len(hist["total"])
print("epochs run:", n)
for frac in [0.25, 0.5, 0.75, 1.0]:
    i = int(n * frac) - 1
    print("  at %5d: total %.6f  chamfer %.6f  pinns %.6f"
          % (i + 1, hist["total"][i], hist["chamfer"][i], hist["pinns"][i]))
last10 = hist["total"][-max(1, n // 10):]
print("plateaued?  last-10%% relative spread = %.3e" % (last10.std() / abs(last10.mean())))


In [ ]:
files = sorted(glob.glob(DATA + "/*.npy"))
zero = [np.sqrt((np.load(os.path.join(GT, os.path.basename(f).replace(".npy",
        "_ground_truth.npy"))) ** 2).sum(1).mean()) for f in files]
ZERO_RMSE = float(np.mean(zero))

model = X.PhysicsInformedRegistration(hidden_dims=X.HIDDEN_DIM_CONFIGS[HD],
                                      activation=ACT, pool_type=POOL).cuda()
model.load_state_dict(torch.load(os.path.join(run_dir, "model_last.pth"), map_location="cuda"))
model.eval()
nparams = sum(p.numel() for p in model.parameters())

errs = []
for f in files:
    a = np.load(f).astype(np.float32)
    gt = np.load(os.path.join(GT, os.path.basename(f).replace(".npy", "_ground_truth.npy")))
    b = torch.tensor(a).unsqueeze(0).cuda()
    pred = (model(b)["deformed_source"] - b[:, 0, :, :]).detach().cpu().numpy()[0]
    errs.append(float(np.sqrt(((pred - gt) ** 2).sum(1).mean())))

res = {"condition": CONDITION, "hidden_dims": HD, "activation": ACT, "pool": POOL,
       "seed": SEED, "epochs": EPOCHS, "params_M": nparams / 1e6,
       "rmse_mean": float(np.mean(errs)), "rmse_per_sample": errs,
       "rmse_predict_zero": ZERO_RMSE,
       "final_total": float(hist["total"][-1]),
       "final_chamfer": float(hist["chamfer"][-1]),
       "final_pinns": float(hist["pinns"][-1])}
with open("/kaggle/working/result.json", "w") as fh:
    json.dump(res, fh, indent=1)

print(json.dumps(res, indent=1))
print()
print("RMSE vs ground truth :", format(res["rmse_mean"], ".6e"))
print("RMSE predicting zero :", format(ZERO_RMSE, ".6e"))
print("BEATS DOING NOTHING? ", res["rmse_mean"] < ZERO_RMSE)
